# Prompt testing

In this notebook, you will explore how the way you write a prompt affects the responses of a chat-tuned LLM. You will:

1. **Set up the connection** to Aitta and choose a chat model to use.
2. **Compare weaker and improved prompts** in three examples and see how small changes in the prompt change the response.
3. **Try few-shot prompting**, where you guide the model by giving it example answers.
4. **Stream a response**, so that you see the text piece by piece while it is being generated.
5. **Experiment on your own** with different models and prompts.

While testing, keep in mind the basic guidelines for effective prompts:

* Be specific
* Provide context
* Define output format
* Use role-based prompts
* Give examples (few-shot prompting)

## Setup

First, we configure the OpenAI client with your Aitta access token and Aitta's base URL. Then we list the models available in Aitta, and you choose one and store its name in the variable `model`. You can continue with `LumiOpen/Llama-Poro-2-70B-Instruct` or choose another chat model, for example `meta-llama/Llama-3.3-70B-Instruct` or `openai/gpt-oss-120b`.

In [2]:
# Set genereated access token here  
access_token = ""

In [3]:
import openai

# Configure the standard OpenAI client to use Aitta's OpenAI-compatible API
client = openai.OpenAI(
    api_key=access_token,
    base_url="https://aitta-api.csc.fi/openai/v1"
)

### List the available models

The list of models comes from Aitta's `/model` endpoint, which is not part of the OpenAI-compatible API, so we call it directly with the `requests` library. The list may also contain embedding models, which cannot be used for chat completions.

Note that models are started on demand. If you choose a model that is currently offline, the first response can take several minutes. Models that are already online (see the [Aitta web frontend](https://aitta.csc.fi)) respond faster.

In [ ]:
import requests

headers = {"Authorization": f"Bearer {access_token}"}
response = requests.get("https://aitta-api.csc.fi/model", headers=headers)
response.raise_for_status()

# The models are listed under "_links" -> "item"; each item has the model id in "name"
available_models = [item["name"] for item in response.json()["_links"]["item"]]
for name in available_models:
    print(name)

In [7]:
# Choose the model to use (copy a name from the list above)
model = ""

### A helper function for sending prompts

To avoid repeating the same code in every example, we define a small helper function `get_response()`. It sends your prompt to the model as a single `user` message and returns the text of the response. In the examples below, you only need to change the prompt.

In [ ]:
# Function to get responses
def get_response(prompt):
    response = client.chat.completions.create(
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        model=model,
        max_completion_tokens=200,
    )
    return response.choices[0].message.content

## Examples of refining prompts

Below are examples demonstrating how to improve prompts for better AI responses. Each case starts with a weaker prompt, followed by an improved version that provides clearer instructions, context, or structure.

Feel free to modify these examples, create your own, and experiment with different models.

### Example 1. 

In [ ]:
prompt = """
Explain machine learning.
"""

response = get_response(prompt)
print(f"Response:\n{response}")

In [ ]:
prompt = """
You are a university professor explaining machine learning to a beginner.
Explain machine learning.
Keep the explanation short, only a few sentences, and don't be too descriptive.
"""

response = get_response(prompt)
print(f"Response:\n{response}")

### Example 2. 

In [ ]:
prompt = """
Explain quantum computing.
"""

response = get_response(prompt)
print(f"Response:\n{response}")

In [ ]:
prompt = """
Explain quantum computing in two sentences, using simple language.
Explain like I would be five years old.
"""


response = get_response(prompt)
print(f"Response:\n{response}")

### Example 3. 

In [ ]:
# https://www.lumi-supercomputer.eu/open-euro-llm/
text = """
A consortium of 20 leading European research institutions, companies and EuroHPC centres coordinated by Jan Hajič (Charles University, Czechia) and co-led by Peter Sarlin (AMD Silo AI, Finland) will build a family of performant, multilingual, large language foundation models for commercial, industrial and public services. 
LUMI will be one of the platforms used in the project. The transparent and compliant open-source models will democratize access to high-quality AI technologies and strengthen the ability of European companies to compete on a global market and public organizations to produce impactful public services.
The OpenEuroLLM project is aligned with the imperative to improve Europe’s competitiveness and digital sovereignty. The project is a prime example of the type of technology infrastructure needed to lower thresholds for European AI product development and refinement, demonstrating the strength of transparency, openness and community involvement, 
values largely recognized across the European tech ecosystem. The models will be developed within Europe’s robust regulatory framework, ensuring alignment with European values while maintaining technological excellence.
Cooperating with open-source and open science communities like LAION, open-sci and OpenML, and additional experts in the field assembled in the project’s Open Strategic Partnership Board, OpenEuroLLM will ensure that the models, software, data and evaluation will be fully open and can be fine-tuned and instruction-tuned for specific industry and 
public sector needs. These performant multilingual models preserve both linguistic and cultural diversity, enabling European companies to develop high-quality products and services in the era of AI.
The project, which has been awarded the STEP (Strategic Technologies for Europe Platform) seal, leverages support from previous European projects and the experience of the partners and their results, including large repositories of high-quality data and pilot LLMs developed previously. The consortium commences its work on February 1st, 2025, with funding from the European Commission under the Digital Europe Programme.
"""

prompt = f"Tell me about this project:\n{text}"

response = get_response(prompt)
print(f"Response:\n{response}")

In [ ]:
better_prompt = f"""Analyze the OpenEuroLLM project described in the following text and provide
A concise summary of the project's main goal with 3-5 bulletpoints.
Text: {text}
"""

response = get_response(better_prompt)
print(f"Response:\n{response}")

### Example 4. 

Let's try **few-shot prompting** next.

Instead of only describing what kind of answer you want, you can **show** it. In few-shot prompting, you write a few example questions and answers yourself and add them to the `messages` list before the actual question. The example answers use the role `assistant`, as if the model had given them. The model then follows the same style and format in its reply.

First, let's ask without examples (this is called *zero-shot* prompting):

In [ ]:
# Zero-shot: only an instruction, no examples
prompt = """Classify the sentiment of the feedback as positive, negative or neutral.
Feedback: The lectures were long but the examples helped."""

response = get_response(prompt)
print(f"Response:\n{response}")

The model probably gives the right idea, but the format of the answer may vary: a full sentence, an explanation, or several words. Now let's add two examples that show the exact format we want.

Note that we cannot use the `get_response()` function here, because it only sends a single `user` message. For few-shot prompting, we build the whole `messages` list ourselves:

In [ ]:
# Few-shot: the same task, but with example answers written by you
messages = [
    {"role": "system", "content": "Classify the sentiment of the feedback as positive, negative or neutral."},
    # example 1, written by you
    {"role": "user", "content": "The course was really useful!"},
    {"role": "assistant", "content": "positive"},
    # example 2, written by you
    {"role": "user", "content": "The exercises did not work."},
    {"role": "assistant", "content": "negative"},
    # the actual question: the model answers in the same one-word format
    {"role": "user", "content": "The lectures were long but the examples helped."}
]

response = client.chat.completions.create(
    model=model,
    messages=messages,
    max_completion_tokens=200
)
print(f"Response:\n{response.choices[0].message.content}")

Compare the two responses. With the examples, the model should answer with just one word, like in the examples. Few-shot prompting is useful e.g. when you want answers in a fixed format that is easy to process further with code.

*Try it:* Add a third example with the answer `neutral`, or change the format of the example answers (e.g. `Sentiment: POSITIVE`) and see if the model follows it.

## Streaming responses

So far, we have received the response only after the model has generated all of it. With the parameter `stream=True`, the response is sent back in small pieces (chunks) while the model is still generating it, so you can see the text appear as it is written. This is how chat applications show the answer word by word, and it is useful especially for long responses: you can start reading right away.

Streaming does not change the content of the response, only how it is delivered. Each chunk contains a small piece of new text in `chunk.choices[0].delta.content`.

In [ ]:
# Stream the response: the text is printed piece by piece as it is generated
stream = client.chat.completions.create(
    model=model,
    messages=[
        {"role": "user", "content": "Write a short story about a reindeer who learns to use a supercomputer."}
    ],
    stream=True   # get the response in chunks instead of all at once
)

for chunk in stream:
    # some chunks contain no text (e.g. the last one), so check before printing
    if chunk.choices and chunk.choices[0].delta.content is not None:
        print(chunk.choices[0].delta.content, end="", flush=True)

## Experiment with different models and prompts

Now that you have completed the examples, you can modify the model or the prompt to observe how these changes affect the output. Feel free to experiment with different configurations and see how the model's responses vary. This is a great way to understand the impact of prompt engineering and model adjustments on the generated text.